In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
%pwd

'/Users/santhoshrishimarkonda/Desktop/Code/UMI/Project 1/nassau-candy-analysis/notebooks'

#### Load Data

In [3]:
df = pd.read_csv('../data/Nassau_Candy_Distributor.csv')
print("Data loaded successfully")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Data loaded successfully
Shape: 10194 rows, 18 columns


In [4]:
print("COLUMN NAMES & DATA TYPES:\n")
print(df.dtypes)

COLUMN NAMES & DATA TYPES:

Row ID              int64
Order ID              str
Order Date            str
Ship Date             str
Ship Mode             str
Customer ID         int64
Country/Region        str
City                  str
State/Province        str
Postal Code           str
Division              str
Region                str
Product ID            str
Product Name          str
Sales             float64
Units               int64
Gross Profit      float64
Cost              float64
dtype: object


#### Null Check

In [5]:
print("MISSING VALUES:\n")
null_counts = df.isnull().sum()
print(null_counts)
print(f"\nTotal missing values: {null_counts.sum()}")

MISSING VALUES:

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
dtype: int64

Total missing values: 0


#### Duplicate Check

> Insight : No missing values in the dataset.

In [6]:
print("DUPLICATE ROWS:\n")
dupes = df.duplicated().sum()
print(f"Duplicate rows found: {dupes}")

DUPLICATE ROWS:

Duplicate rows found: 0


> Insight : No duplicate values in the dataset.

#### Date Range

In [7]:
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)

print("DATE RANGE:\n")
print(f"Order Date : {df['Order Date'].min().date()}  to  {df['Order Date'].max().date()}")
print(f"Ship Date  : {df['Ship Date'].min().date()}  to  {df['Ship Date'].max().date()}")

DATE RANGE:

Order Date : 2024-01-02  to  2025-12-31
Ship Date  : 2026-06-30  to  2030-06-28


> "FLAGGED" — Ship dates range from 2026 to 2030, postdating every order.
>
> "FLAGGED" — Order Date itself spans two calendar years (2024-01-02 to 2025-12-31), not just 2025. Analysis scope will be restricted to FY2025 only during cleaning (notebook 02) so monthly/quarterly trends aren't a blend of two years.

#### Unique Values

In [8]:
print("UNIQUE VALUE COUNTS\n")
cols = ['Division', 'Product Name', 'Region', 'Ship Mode', 'Country/Region']
for col in cols:
    print(f"{col}: {df[col].nunique()} unique:")
    for unique_prods in df[col].unique().tolist():
        print(unique_prods)
    print("")

UNIQUE VALUE COUNTS

Division: 3 unique:
Chocolate
Other
Sugar

Product Name: 15 unique:
Wonka Bar - Milk Chocolate
Wonka Bar - Triple Dazzle Caramel
Wonka Bar - Nutty Crunch Surprise
Wonka Bar -Scrumdiddlyumptious
Wonka Bar - Fudge Mallows
Wonka Gum
Kazookles
Lickable Wallpaper
Fizzy Lifting Drinks
Laffy Taffy
SweeTARTS
Nerds
Hair Toffee
Everlasting Gobstopper
Fun Dip

Region: 4 unique:
Interior
Atlantic
Gulf
Pacific

Ship Mode: 4 unique:
Standard Class
First Class
Second Class
Same Day

Country/Region: 2 unique:
United States
Canada



In [9]:
print("DESCRIPTIVE STATS:\n")
df[['Sales', 'Units', 'Gross Profit', 'Cost']].describe().round(2)

DESCRIPTIVE STATS:



,Sales,Units,Gross Profit,Cost
count,10194.00,10194.00,10194.00,10194.00
mean,13.91,3.79,9.17,4.74
std,11.34,2.23,6.64,5.06
min,1.25,1.00,0.25,0.60
25%,7.20,2.00,4.90,2.40
50%,10.80,3.00,7.47,3.60
75%,18.00,5.00,12.25,5.70
max,260.00,14.00,130.00,130.00


#### Verify Sales - Cost = Gross Profit

In [10]:
print("DATA INTEGRITY CHECK:\n")
df['Calculated_Profit'] = df['Sales'] - df['Cost']
df['Profit_Match'] = np.isclose(df['Calculated_Profit'], df['Gross Profit'], atol=0.01)

mismatches = df[~df['Profit_Match']]
print(f"Rows where Sales - Cost ≠ Gross Profit: {len(mismatches)}")

if len(mismatches) == 0:
    print("Yes, All rows verified — data is internally consistent")
else:
    print("No, Mismatches found:")
    print(mismatches[['Sales', 'Cost', 'Gross Profit', 'Calculated_Profit']].head(10))

# Clean up helper columns
df.drop(columns=['Calculated_Profit', 'Profit_Match'], inplace=True)

DATA INTEGRITY CHECK:

Rows where Sales - Cost ≠ Gross Profit: 0
Yes, All rows verified — data is internally consistent


#### Summary

In [11]:
print("DATASET SUMMARY:\n")
print(f"Total Rows        : {len(df):,}")
print(f"Total Orders      : {df['Order ID'].nunique():,}")
print(f"Total Products    : {df['Product Name'].nunique()}")
print(f"Divisions         : {df['Division'].nunique()}")
print(f"Regions           : {df['Region'].nunique()}")
print(f"Date Range        : {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")
print(f"Null Values       : {df.isnull().sum().sum()}")
print(f"Duplicate Rows    : {df.duplicated().sum()}")
print(f"Data Integrity    : Verified")

DATASET SUMMARY:

Total Rows        : 10,194
Total Orders      : 8,549
Total Products    : 15
Divisions         : 3
Regions           : 4
Date Range        : 2024-01-02 to 2025-12-31
Null Values       : 0
Duplicate Rows    : 0
Data Integrity    : Verified


#### Report

In [12]:
report = {
    "dataset_overview": {
        "total_rows": len(df),
        "total_columns": df.shape[1],
        "total_orders": df['Order ID'].nunique(),
        "total_products": df['Product Name'].nunique(),
        "total_divisions": df['Division'].nunique(),
        "total_regions": df['Region'].nunique(),
        "date_range": {
            "order_start": str(df['Order Date'].min().date()),
            "order_end": str(df['Order Date'].max().date()),
            "ship_start": str(df['Ship Date'].min().date()),
            "ship_end": str(df['Ship Date'].max().date())
        }
    },
    "data_quality": {
        "null_values": int(df.isnull().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "profit_integrity_check": "PASSED — Sales - Cost = Gross Profit on all rows",
        "ship_date_anomaly": "FLAGGED — Ship dates range from 2026 to 2030, postdating every order. Ship date column is unreliable and will be excluded from time-based analysis.",
        "multi_year_order_data": (
            f"FLAGGED — Order Date spans {df['Order Date'].min().date()} to "
            f"{df['Order Date'].max().date()}, covering both 2024 and 2025. "
            "Analysis scope is restricted to FY2025 only during cleaning (notebook 02)."
        )
    },
    "unique_values": {
        "divisions": df['Division'].unique().tolist(),
        "products": df['Product Name'].unique().tolist(),
        "regions": df['Region'].unique().tolist(),
        "ship_modes": df['Ship Mode'].unique().tolist()
    },
    "basic_statistics": {
        "sales": df['Sales'].describe().round(2).to_dict(),
        "units": df['Units'].describe().round(2).to_dict(),
        "gross_profit": df['Gross Profit'].describe().round(2).to_dict(),
        "cost": df['Cost'].describe().round(2).to_dict()
    }
}

In [13]:
import json
import os

output_path = '../outputs/reports/data_understanding_report.json'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(report, f, indent=4)

print(f"Report saved to: {output_path}")

Report saved to: ../outputs/reports/data_understanding_report.json
